<a href="https://colab.research.google.com/github/wtree101/ZIP-RC-Colab/blob/main/notebooks/colab/03_training_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 3 — 正式训练数据生成与标注

在 Pilot 通过后生成 `2,000 prompts × 2 = 4,000 trajectories`，并写入 correctness 标签。

本阶段再次检查截断和标签平衡，防止规模扩大后数据分布发生变化。

先运行 `colab/00_memory_and_config.ipynb`；本 Notebook 的训练命令会自动使用 ZIP mamba 环境。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path("/content/ZIP-RC-Colab")
ZIP_PY = Path("/content/mamba/envs/zip/bin/python")

if not REPO.exists():
    raise FileNotFoundError("远端仓库不存在；请先运行 colab/00_memory_and_config.ipynb。")
if not ZIP_PY.exists():
    raise FileNotFoundError("ZIP Python 环境不存在；请先运行 colab/00_memory_and_config.ipynb。")

os.environ["ZIPRC_PYTHON"] = str(ZIP_PY)
sys.path.insert(0, str(REPO / "notebooks"))
from ziprc_notebook_utils import *

CONFIG = load_config(REPO)
print("Repository:", REPO)
print("ZIP Python:", ZIP_PY)
print("Experiment:", CONFIG["experiment_name"])

In [ ]:
RUN_STAGE = True
full_path = REPO / CONFIG["paths"]["full"]
full_grader_metrics = REPO / "artifacts/metrics/full_grader.json"
if RUN_STAGE:
    run_repo(
        REPO,
        "python3", "src/generate_ziprc_rollouts.py",
        "--model", CONFIG["model_id"], "--dataset", CONFIG["dataset"],
        "--split", CONFIG["split"], "--prompt-column", CONFIG["prompt_column"],
        "--answer-column", CONFIG["answer_column"], "--out", full_path,
        "--max-num-prompts", CONFIG["training_prompts"],
        "--thinking-samples", 0, "--non-thinking-samples", CONFIG["training_rollouts_per_prompt"],
        "--temperature", CONFIG["temperature"], "--min-p", CONFIG["min_p"],
        "--max-model-len", CONFIG["generation_max_model_len"],
        "--max-new-tokens", CONFIG["max_output_tokens"],
        "--max-num-seqs", CONFIG["max_num_seqs"], "--dtype", CONFIG["dtype"],
        "--dp-size", 1, "--tp-size", 1,
    )
    run_repo(
        REPO,
        "python3", "src/evaluate_and_label_rollouts.py",
        "--data", full_path, "--model", CONFIG["grader_model_id"],
        "--tensor-parallel-size", 1, "--gpu-memory-utilization", CONFIG["gpu_memory_utilization"],
        "--max-model-len", CONFIG["grader_max_model_len"], "--max-num-seqs", CONFIG["max_num_seqs"],
        "--dtype", CONFIG["dtype"], "--output-json", full_grader_metrics,
    )

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

df = pd.read_parquet(full_path)
expected = int(CONFIG["training_prompts"]) * int(CONFIG["training_rollouts_per_prompt"])
accuracy = float(df["correct"].mean())
finished_rate = float(df["finished"].mean())
cap_rate = float((df["length"] >= CONFIG["max_output_tokens"]).mean())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df["length"], bins=40, color="#4c78a8")
axes[0].axvline(CONFIG["max_output_tokens"], color="#e45756", linestyle="--")
axes[0].set(title="Full-data response length", xlabel="tokens")
df["correct"].value_counts().sort_index().plot.bar(ax=axes[1], color=["#e45756", "#49beaa"])
axes[1].set_title("Correct / incorrect samples")
per_prompt = df.groupby("prompt_idx").agg(correct_rate=("correct", "mean"), mean_length=("length", "mean"))
axes[2].scatter(per_prompt["mean_length"], per_prompt["correct_rate"], s=10, alpha=.35)
axes[2].set(title="Per-prompt difficulty", xlabel="mean output tokens", ylabel="correct rate")
plt.tight_layout()
plt.show()

checks = [
    gate("样本数完整", len(df) == expected, f"{len(df)}/{expected}"),
    gate("Prompt 数完整", df["prompt_idx"].nunique() == CONFIG["training_prompts"], f"{df['prompt_idx'].nunique()} prompts"),
    gate("Finished rate ≥95%", finished_rate >= .95, f"{finished_rate:.1%}", kind="scientific"),
    gate("截断 <5%", cap_rate < .05, f"{cap_rate:.1%}", kind="scientific"),
    gate("正负标签可学习", df["correct"].nunique() == 2 and df["correct"].value_counts(normalize=True).min() >= .10, f"accuracy={accuracy:.1%}", kind="scientific"),
]
display(gate_frame(checks))
save_stage_report(REPO, "03_training_data", checks, {"rows": len(df), "prompts": int(df['prompt_idx'].nunique()), "accuracy": accuracy, "finished_rate": finished_rate, "cap_rate": cap_rate})